<a href="https://colab.research.google.com/github/Aya-Elgammal/computer_vision-/blob/main/Cats_vs_Dogs_Classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
data_dir = "/content/drive/MyDrive/PetImages"

In [7]:
os.listdir(data_dir)

['Dog', 'Cat']

In [5]:
os.listdir(data_dir + "/Cat")[:5]

['8952.jpg', '9068.jpg', '9208.jpg', '8916.jpg', '9.jpg']

In [8]:
filepaths = []
labels = []

folds = os.listdir(data_dir)

for fold in folds:
    foldpath = os.path.join(data_dir, fold)
    files = os.listdir(foldpath)

    for file in files:
        filepath = os.path.join(foldpath, file)
        filepaths.append(filepath)
        labels.append(fold)

df = pd.DataFrame({
    'filepaths': filepaths,
    'labels': labels
})

df.head()

,filepaths,labels
0,/content/drive/MyDrive/PetImages/Dog/9368.jpg,Dog
1,/content/drive/MyDrive/PetImages/Dog/9313.jpg,Dog
2,/content/drive/MyDrive/PetImages/Dog/9044.jpg,Dog
3,/content/drive/MyDrive/PetImages/Dog/9241.jpg,Dog
4,/content/drive/MyDrive/PetImages/Dog/9001.jpg,Dog


In [9]:
print("Cats:", len(os.listdir("/content/drive/MyDrive/PetImages/Cat")))
print("Dogs:", len(os.listdir("/content/drive/MyDrive/PetImages/Dog")))

Cats: 12499
Dogs: 12499


In [10]:
train_df,test_df = train_test_split(df, test_size = 0.2, random_state = 42)

In [11]:
gen = ImageDataGenerator()
train_gen = gen.flow_from_dataframe(train_df,x_col='filepaths',y_col='labels',target_size=(224,224),
                                    class_mode='binary',color_mode='rgb',batch_size=16)
test_gen = gen.flow_from_dataframe(test_df,x_col='filepaths',y_col='labels',target_size=(224,224),
                                    class_mode='binary',color_mode='rgb',batch_size=16)

Found 19998 validated image filenames belonging to 2 classes.
Found 5000 validated image filenames belonging to 2 classes.


In [16]:
base_model = tf.keras.applications.EfficientNetB3(include_top=False,input_shape=(224,224,3))
model = Sequential([
    base_model,
    Flatten(),
    Dense(128,activation='relu'),
    Dense(64,activation='relu'),

    Dense(1,activation='sigmoid')
])
model.compile(optimizer=Adam(learning_rate=0.0001), loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb3 (Functional)     │ (None, 7, 7, 1536)     │    10,783,535 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 75264)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 128)            │     9,633,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,425,776 (77.92 MB)

 Trainable params: 20,338,473 (77.59 MB)

 Non-trainable params: 87,303 (341.03 KB)

In [12]:
model = Sequential()

model.add(Conv2D(32,(3,3),activation='relu',input_shape=(224,224,3)))
model.add(MaxPooling2D(2,2))

model.add(Conv2D(64,(3,3),activation='relu'))
model.add(MaxPooling2D(2,2))

model.add(Conv2D(128,(3,3),activation='relu'))
model.add(MaxPooling2D(2,2))

model.add(Flatten())

model.add(Dense(128,activation='relu'))
model.add(Dropout(0.5))

model.add(Dense(1,activation='sigmoid'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [13]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [14]:
history = model.fit(
    train_gen,
    validation_data=test_gen,
    epochs=10
)

Epoch 1/10
1024/1250 ━━━━━━━━━━━━━━━━━━━━ 10:13 3s/step - accuracy: 0.5089 - loss: 7.4823

/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


1250/1250 ━━━━━━━━━━━━━━━━━━━━ 4151s 3s/step - accuracy: 0.5087 - loss: 1.5704 - val_accuracy: 0.5070 - val_loss: 0.6933
Epoch 2/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 115s 92ms/step - accuracy: 0.5159 - loss: 0.6980 - val_accuracy: 0.5218 - val_loss: 0.6961
Epoch 3/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 114s 92ms/step - accuracy: 0.5308 - loss: 0.6887 - val_accuracy: 0.5174 - val_loss: 0.6979
Epoch 4/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 113s 90ms/step - accuracy: 0.5365 - loss: 0.6843 - val_accuracy: 0.5184 - val_loss: 0.7048
Epoch 5/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 114s 91ms/step - accuracy: 0.5519 - loss: 0.6728 - val_accuracy: 0.5346 - val_loss: 0.7113
Epoch 6/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 112s 90ms/step - accuracy: 0.5614 - loss: 0.6710 - val_accuracy: 0.4950 - val_loss: 0.7134
Epoch 7/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 114s 91ms/step - accuracy: 0.5484 - loss: 0.6917 - val_accuracy: 0.5132 - val_loss: 0.7155
Epoch 8/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 113s 90ms/step - accuracy: 0.5499 - los

In [17]:
model.fit(train_gen,epochs=10)

Epoch 1/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 402s 215ms/step - accuracy: 0.9661 - loss: 0.0898
Epoch 2/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 197s 158ms/step - accuracy: 0.9919 - loss: 0.0255
Epoch 3/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 198s 158ms/step - accuracy: 0.9951 - loss: 0.0152
Epoch 4/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 197s 158ms/step - accuracy: 0.9946 - loss: 0.0185
Epoch 5/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 197s 158ms/step - accuracy: 0.9956 - loss: 0.0148
Epoch 6/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 198s 159ms/step - accuracy: 0.9964 - loss: 0.0106
Epoch 7/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 200s 160ms/step - accuracy: 0.9968 - loss: 0.0094
Epoch 8/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 199s 159ms/step - accuracy: 0.9971 - loss: 0.0098
Epoch 9/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 199s 159ms/step - accuracy: 0.9966 - loss: 0.0108
Epoch 10/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 199s 159ms/step - accuracy: 0.9974 - loss: 0.0086


In [18]:
model.evaluate(test_gen)

313/313 ━━━━━━━━━━━━━━━━━━━━ 53s 132ms/step - accuracy: 0.9890 - loss: 0.0681


[0.06810235977172852, 0.9890000224113464]

In [19]:
model.save("/content/drive/MyDrive/cat_dog_model.h5")

In [34]:
from tensorflow.keras.preprocessing import image
img_path = "/content/test1.jpg"

img = image.load_img(img_path,target_size=(224,224))
img_array = image.img_to_array(img)

img_array = np.expand_dims(img_array,axis=0)
img_array = img_array/255.0

pred = model.predict(img_array)

if pred > 0.5:
    print("Dog")
else:
    print("Cat")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step
Cat


In [35]:
!pip install fastapi uvicorn nest-asyncio pyngrok tensorflow pillow python-multipart

In [36]:
import gradio as gr
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
import numpy as np
from PIL import Image

model = load_model("/content/drive/MyDrive/cat_dog_model.h5")

def predict_image(img):
    try:
        img = img.convert("RGB")
        img = img.resize((224, 224))
        img_array = image.img_to_array(img)
        img_array = np.expand_dims(img_array, axis=0)
        img_array = img_array / 255.0

        pred = model.predict(img_array)[0][0]

        if pred > 0.5:
            return "Dog "
        else:
            return "Cat "
    except Exception as e:
        return f"Error: {str(e)}"

iface = gr.Interface(
    fn=predict_image,
    inputs=gr.Image(type="pil"),
    outputs="text",
    live=False,
    title="Cat vs Dog Classifier",
    description="Upload an image of a cat or dog and get the prediction."
)

iface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d589c2fbf5a1c190ac.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
